(ex-benchmarking)=

# Benchmarking and `cotengra`

If you would like to benchmark against `cotengra` in a publication it would be appreciated to take care in representing the performance accurately. Here we demonstrate reasonable optimizer settings that might be used, with the example of a depth-20 sycamore circuit amplitude.

Other notes:

1. The following searches for *unsliced* paths (i.e. with no memory constraint), these give a lower cost limit for the cost of a sliced contraction.
2. This is for a single amplitude. For a random quantum circuit XEB run for example, one might have to compute $\sim 1,000,000$ amplitudes.
3. Remove refining options to get the original performance of https://arxiv.org/abs/2002.01935.
4. Directly minimizing `flops` does not yield good hardware performance, for that use `minimize="combo"`.

In [1]:
# check for rust acceleration
import cotengrust as ctgr
import quimb.tensor as qtn

import cotengra as ctg

## Generate a tensor network

In [3]:
circ = qtn.Circuit.from_qsim_file("circuit_n53_m20_s0_e0_pABCDCDAB.qsim")

In [4]:
tn = circ.amplitude_tn()
tn

TensorNetworkGenVector(tensors=381, indices=754)

## `random-greedy`

The random greedy optimizer is very efficient in terms of time to get a 'good' solution, especially with `cotengrust` installed.

In [5]:
opt = ctg.RandomGreedyOptimizer(max_repeats=128, seed=0)

In [6]:
%%time
tree = tn.contraction_tree(opt)

CPU times: user 1.41 s, sys: 319 ms, total: 1.73 s
Wall time: 150 ms


In [7]:
tree.contraction_cost(log=10)

18.680759440930593

The only parameter is `max_repeats`. (Note there is a sub-linear dependence on this as once we know a good score the optimizer starts to terminate other samples early).

## `kahypar`

For hard problems, the hyper optimized `greedy` + `kahypar` approach is recommended.

In [8]:
opt = ctg.HyperOptimizer(
    methods=["greedy", "kahypar"],
    max_time=5 * 60,  # 5 minutes max
    max_repeats=1024,
    reconf_opts={
        "subtree_size": 12
    },  # use high quality subtree reconfiguration
    optlib="optuna",
    parallel=16,  # number of parallel processes to use
    progbar=True,
)

In [9]:
tree = tn.contraction_tree(opt)

F=18.27 C=19.35 S=52.00 P=53.32:  55%|███████████████████████████████▌                         | 567/1024 [05:00<04:01,  1.89it/s]


In [10]:
tree.contraction_cost(log=10)

18.270356493772695

You may have to tune `subtree_size`, `maxiter` and `select` within `reconf_opts`. The tradeoff is between sampling more trees or spending more time refining each one.

## `simulated_annealing`

For some problems, simulated annealing (as described in https://arxiv.org/abs/2108.05665) might be best:

In [11]:
opt = ctg.HyperOptimizer(
    max_time=5 * 60,  # 5 minutes max
    max_repeats=1024,
    simulated_annealing_opts={},
    reconf_opts={"subtree_size": 6},  # combine with subtree reconfiguration
    optlib="optuna",
    parallel=16,  # number of parallel processes to use
    progbar=True,
)

In [12]:
tree = tn.contraction_tree(opt)

F=18.04 C=19.18 S=52.00 P=53.00:  24%|█████████████▋                                           | 246/1024 [05:01<15:54,  1.23s/it]


In [13]:
tree.contraction_cost(log=10)

18.039147512279065

The main parameter to tune within `simulated_annealing_opts` is `tsteps` which controls the total annealing time. Again, the tradeoff is between sampling more trees or spending more time refining each one.

## with slicing (constrained memory)

You will want to use `slicing_reconf_opts` or set `target_size` in `simulated_annealing_opts` for high quality slicing - see the main docs. `slicing_opts` is the original method of https://arxiv.org/abs/2002.01935 and useful for moderate slicing only.